# HC3 — Fixed Logistic Regression Pipeline

## What was wrong and why

The original pipeline achieved 0.98 on a held-out test split but failed on real-world text.
That gap between benchmark accuracy and real-world performance is the definition of overfitting.
The causes were:

1. **`spelling_error_ratio` is a constant (always 1.0)** — confirmed in the output. A constant
   feature carries zero information but adds noise to the scaler and confuses the model.
   Dropped.

2. **Raw count features leak document length** — `punct_count`, `function_count`,
   `discourse_count`, `chat_word_count` are all absolute counts. Longer answers naturally
   have more of everything. AI answers in HC3 tend to be a consistent length, so the model
   learns *length* as a proxy rather than *style*. On real data, humans also write long
   answers, so this proxy breaks. **Replaced with ratios** (already computed).

3. **TF-IDF with 10,000 features and no `max_df` cap** — the vectoriser memorises
   HC3-specific phrases ("as an AI language model", "certainly", "in conclusion") that are
   dataset artefacts of 2023-era ChatGPT. Modern AI and human text both use these phrases.
   **Fixed with `max_df=0.85`** (drop terms appearing in >85 % of docs — they are
   stop-words) and **reduced vocabulary to 5,000** to force the model to learn patterns
   rather than memorise signatures.

4. **No regularisation search** — `C=1.0` is arbitrary. A high C means low regularisation,
   which lets the model fit training noise. **Fixed with `LogisticRegressionCV`**, which
   performs stratified cross-validated C selection using the training data only. No extra
   model — this is still logistic regression.

5. **No cross-validation score reported** — a single split gives one number that can be
   lucky. **Fixed with 5-fold stratified cross-validation** on the full dataset to report
   mean ± std, which tells you how much the model's performance varies.

6. **Train/test inference mismatch** — training was done on whole answers, but inference
   used 3-sentence chunks with probability averaging. The custom feature distributions
   (e.g. `sentence_length_mean`) computed on a 3-sentence chunk are completely different
   from those computed on a 15-sentence answer. **Fixed: inference now mirrors training
   exactly** — pass the full text, extract features once, predict once.

**Everything else is kept as-is.** The preprocessing pipeline, the feature engineering
(spaCy sentenciser, word lists), the TF-IDF + custom feature fusion, and logistic
regression are all sound.

---
## Step 0 — Imports

In [2]:
import ast
import joblib
import numpy as np
import pandas as pd
import spacy
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42

---
## Step 1 — Load cleaned data

In [3]:
df = pd.read_csv('cleaned_data.csv')

print(f'Shape: {df.shape}')
print(f'Nulls:\n{df.isnull().sum()}')
print(f'Duplicates: {df.duplicated().sum()}')
print(f'\nLabel distribution:')
print(df['label'].value_counts())
df.sample(5)

Shape: (45750, 2)
Nulls:
answers    0
label      0
dtype: int64
Duplicates: 0

Label distribution:
label
1    23304
0    22446
Name: count, dtype: int64


,answers,label
26509,"[""There are a few different reasons why some p...",1
44886,"[""It is not uncommon for some individuals to e...",1
2351,['My time to shine . Bear with me as i \'m on ...,0
18852,"[""What can you do? Pay the loan or face the d...",0
9416,['Personally A ) to be taller B)makes calf mus...,0


---
## Step 2 — Text cleaning

The raw `cleaned_data.csv` may still contain un-exploded list strings
(rows that look like `['answer text here']`). We parse those out and lowercase.

In [4]:
def clean_text(x):
    """Unwrap serialised list strings; return plain lowercase string."""
    if isinstance(x, list):
        return ' '.join(map(str, x))
    if isinstance(x, dict):
        return ' '.join(map(str, x.values()))
    if isinstance(x, str):
        try:
            parsed = ast.literal_eval(x)
            if isinstance(parsed, list):
                return ' '.join(map(str, parsed))
            return str(parsed)
        except (ValueError, SyntaxError):
            return x
    return str(x)


df['answers'] = df['answers'].str.lower().apply(clean_text)

print('Text cleaning applied.')
print(df['answers'].head(3).tolist())

Text cleaning applied.
['basically there are many categories of " best seller " . replace " best seller " by something like " oscars " and every " best seller " book is basically an " oscar - winning " book . may not have won the " best film " , but even if you won the best director or best script , you \'re still an " oscar - winning " film . same thing for best sellers . also , iirc the rankings change every week or something like that . some you might not be best seller one week , but you may be the next week . i guess even if you do n\'t stay there for long , you still achieved the status . hence , # 1 best seller .if you \'re hearing about it , it \'s because it was a very good or very well - publicized book ( or both ) , and almost every good or well - publicized book will be # 1 on the ny times bestseller list for at least a little bit . kindof like how almost every big or good movies are # 1 at the box office on their opening weekend .one reason is lots of catagories . however 

---
## Step 3 — Feature extraction

### Key change: drop all raw count features, keep only ratios

Raw counts (`punct_count`, `function_count`, `discourse_count`, `chat_word_count`) correlate
with text length, not with style. Since AI and human answers have different length
distributions in HC3, the model learns to exploit length rather than language style.
On real data where a human writes a long, structured answer, the model predicts AI.

We keep: `ttr`, `function_word_ratio`, `discourse_ratio`, `chat_word_ratio`,
`punct_ratio`, `sentence_length_mean`, `sentence_length_std`, `avg_word_length`.
These are all length-normalised and transfer to out-of-domain text.

`spelling_error_ratio` is dropped — it was always 1.0 (confirmed in your outputs).

In [5]:
nlp = spacy.load('en_core_web_sm', disable=['ner', 'parser'])
nlp.add_pipe('sentencizer')


def load_word_list(path):
    with open(path, 'r', encoding='utf-8') as f:
        return set(line.strip().lower() for line in f if line.strip())


chat_words       = load_word_list('chat_words.txt')
function_words   = load_word_list('function_words.txt')
discourse_markers = load_word_list('discourse_markers.txt')


def split_sentences(text):
    doc = nlp(text)
    return [s.text.strip() for s in doc.sents if s.text.strip()]


def extract_features(text):
    """Extract sentence-level features. Returns length-normalised ratios only."""
    doc = nlp(text)
    tokens      = [t.text.lower() for t in doc if not t.is_space]
    alpha_tokens = [t.text.lower() for t in doc if t.is_alpha]

    total_tokens = len(tokens)
    total_alpha  = len(alpha_tokens)

    zero = {
        'chat_word_ratio'    : 0.0,
        'punct_ratio'        : 0.0,
        'ttr'                : 0.0,
        'function_word_ratio': 0.0,
        'discourse_ratio'    : 0.0,
        'sentence_length'    : 0.0,
        'avg_word_length'    : 0.0,
    }

    if total_tokens == 0 or total_alpha == 0:
        return zero

    # --- ratios only, no raw counts ---
    chat_count      = sum(1 for t in tokens if t in chat_words)
    punct_count     = sum(1 for t in doc if t.is_punct)
    function_count  = sum(1 for t in tokens if t in function_words)
    discourse_count = sum(1 for t in tokens if t in discourse_markers)

    return {
        'chat_word_ratio'    : chat_count     / total_tokens,
        'punct_ratio'        : punct_count    / total_tokens,
        'ttr'                : len(set(alpha_tokens)) / total_alpha,
        'function_word_ratio': function_count  / total_tokens,
        'discourse_ratio'    : discourse_count / total_tokens,
        'sentence_length'    : float(total_tokens),
        'avg_word_length'    : sum(len(t) for t in alpha_tokens) / total_alpha,
    }


def aggregate_sentence_features(sentences):
    """Aggregate sentence-level features to document level."""
    feats = [extract_features(s) for s in sentences]
    if not feats:
        return {}
    feat_df = pd.DataFrame(feats)
    return {
        'chat_word_ratio'    : feat_df['chat_word_ratio'].mean(),
        'punct_ratio'        : feat_df['punct_ratio'].mean(),
        'ttr'                : feat_df['ttr'].mean(),
        'function_word_ratio': feat_df['function_word_ratio'].mean(),
        'discourse_ratio'    : feat_df['discourse_ratio'].mean(),
        'sentence_length_mean': feat_df['sentence_length'].mean(),
        'sentence_length_std' : feat_df['sentence_length'].std(ddof=1) if len(feat_df) > 1 else 0.0,
        'avg_word_length'    : feat_df['avg_word_length'].mean(),
    }


def build_features(df_in):
    rows = []
    for _, row in df_in.iterrows():
        sents = split_sentences(row['answers'])
        feats = aggregate_sentence_features(sents)
        feats['label'] = row['label']
        rows.append(feats)
    return pd.DataFrame(rows)


print('Functions defined. Running feature extraction (this takes a few minutes)...')
df_feat = build_features(df)
print(f'Feature matrix shape: {df_feat.shape}')
df_feat.head()

Functions defined. Running feature extraction (this takes a few minutes)...
Feature matrix shape: (45750, 9)


,chat_word_ratio,punct_ratio,ttr,function_word_ratio,discourse_ratio,sentence_length_mean,sentence_length_std,avg_word_length,label
0,0.022145,0.177109,0.899103,0.442874,0.059134,24.272727,14.185139,4.373566,0
1,0.004545,0.116720,0.918157,0.458618,0.049649,20.750000,10.289877,4.537067,0
2,0.008328,0.127167,0.900015,0.391254,0.042689,36.923077,23.514044,4.643522,0
3,0.006944,0.141427,0.933176,0.415362,0.055521,22.111111,19.303137,4.923874,0
4,0.007407,0.104528,0.868949,0.512040,0.050100,31.666667,12.549900,4.692924,0


### Quick sanity check — feature distributions by label

If the features have discriminative power, their means should differ between label 0 and label 1.

In [6]:
feature_cols_custom = [c for c in df_feat.columns if c != 'label']

print('--- Mean feature values by label ---')
print(df_feat.groupby('label')[feature_cols_custom].mean().T.rename(columns={0: 'Human (0)', 1: 'AI (1)'}).round(4))

--- Mean feature values by label ---
label                 Human (0)   AI (1)
chat_word_ratio          0.0031   0.0020
punct_ratio              0.1325   0.0996
ttr                      0.9024   0.8678
function_word_ratio      0.4481   0.4799
discourse_ratio          0.0422   0.0459
sentence_length_mean    25.0494  29.6330
sentence_length_std     13.2496  12.2825
avg_word_length          4.5471   4.6266


---
## Step 4 — Assemble full feature dataframe and split

We join the text column back for TF-IDF, then split **once** into train/test.
All fitting (TF-IDF, scaler, model) happens on training data only.

In [7]:
df_full = pd.concat(
    [df[['answers', 'label']].reset_index(drop=True),
     df_feat.drop(columns=['label']).reset_index(drop=True)],
    axis=1
)

X_ans    = df_full['answers']
y        = df_full['label']
X_custom = df_full.drop(columns=['answers', 'label'])

print(f'Custom feature columns: {X_custom.columns.tolist()}')

# Stratified split — keeps class ratio identical in train and test
X_ans_train, X_ans_test, X_custom_train, X_custom_test, y_train, y_test = train_test_split(
    X_ans, X_custom, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

print(f'Train size: {len(y_train)}  Test size: {len(y_test)}')
print(f'Train label balance: {y_train.value_counts().to_dict()}')

Custom feature columns: ['chat_word_ratio', 'punct_ratio', 'ttr', 'function_word_ratio', 'discourse_ratio', 'sentence_length_mean', 'sentence_length_std', 'avg_word_length']
Train size: 36600  Test size: 9150
Train label balance: {1: 18643, 0: 17957}


---
## Step 5 — TF-IDF vectorisation

### Key changes vs original:

- `max_features` reduced from 10,000 → **5,000**. A smaller vocabulary means the model
  cannot memorise low-frequency HC3 phrases and must learn robust patterns.
- `max_df=0.85` added. Terms appearing in more than 85 % of documents are effectively
  stop-words for classification — they don't separate the classes.
- `min_df=5` (was 3). Slightly higher threshold filters out more corpus-specific rare terms.
- Everything else unchanged: `ngram_range=(1,2)`, `sublinear_tf=True`.

In [8]:
tfidf = TfidfVectorizer(
    max_features=5_000,       # reduced: forces generalisation
    ngram_range=(1, 2),
    min_df=5,                 # slightly higher: filters corpus-specific rarities
    max_df=0.85,              # new: removes near-universal terms
    sublinear_tf=True,
)

# Fit ONLY on training data — no leakage
X_train_tfidf = tfidf.fit_transform(X_ans_train)
X_test_tfidf  = tfidf.transform(X_ans_test)

print(f'TF-IDF vocabulary size: {len(tfidf.vocabulary_)}')
print(f'Train TF-IDF shape: {X_train_tfidf.shape}')

TF-IDF vocabulary size: 5000
Train TF-IDF shape: (36600, 5000)


---
## Step 6 — Scale custom features

Fill NaN (single-sentence std = NaN by design) and replace any inf values
before scaling. Scaler is fit on training data only.

In [9]:
X_custom_train = X_custom_train.replace([np.inf, -np.inf], 0).fillna(0)
X_custom_test  = X_custom_test.replace([np.inf, -np.inf], 0).fillna(0)

scaler = StandardScaler(with_mean=False)
X_train_custom_scaled = scaler.fit_transform(X_custom_train)
X_test_custom_scaled  = scaler.transform(X_custom_test)

print('Custom features scaled.')
print(f'Custom feature shape: {X_train_custom_scaled.shape}')

Custom features scaled.
Custom feature shape: (36600, 8)


---
## Step 7 — Fuse features

In [10]:
X_train = hstack([X_train_tfidf, X_train_custom_scaled])
X_test  = hstack([X_test_tfidf,  X_test_custom_scaled])

print(f'Final train feature matrix: {X_train.shape}')
print(f'Final test feature matrix:  {X_test.shape}')

Final train feature matrix: (36600, 5008)
Final test feature matrix:  (9150, 5008)


---
## Step 8 — Cross-validated logistic regression

### Key change: `LogisticRegressionCV` instead of `LogisticRegression(C=1.0)`

`LogisticRegressionCV` is still logistic regression — it just searches for the best
regularisation strength `C` using stratified cross-validation **on the training set**.
This means:

- The test set is never touched during C selection (no leakage)
- The model is not over-regularised (underfit) or under-regularised (overfit)
- `Cs=10` tries 10 logarithmically spaced values of C
- `cv=5` uses 5-fold stratified cross-validation internally
- `solver='saga'` is required for L1/L2 with large sparse matrices
- `penalty='l2'` — L2 regularisation shrinks all weights, preventing the model from
  putting too much weight on HC3-specific phrases

The winning `C` value is printed below — if it is very high (> 10), regularisation had
little effect; if it is low (< 0.01), the model needed strong regularisation.

In [11]:
model = LogisticRegressionCV(
    Cs=10,                       # 10 candidates on log scale from 1e-4 to 1e4
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    penalty='l2',
    solver='saga',
    max_iter=2000,
    class_weight='balanced',
    scoring='f1_macro',          # optimise for balance between classes, not just accuracy
    n_jobs=-1,
    random_state=RANDOM_STATE,
)

model.fit(X_train, y_train)

best_C = model.C_[0]
print(f'Best C selected by cross-validation: {best_C:.6f}')
if best_C > 10:
    print('Note: high C — model needed little regularisation. TF-IDF restriction is doing the work.')
elif best_C < 0.01:
    print('Note: low C — strong regularisation applied. Features may have high variance.')
else:
    print('Note: moderate C — regularisation is well-balanced.')

/usr/local/python/3.12.1/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1780: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
/usr/local/python/3.12.1/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1811: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratios' instead. Use l1_ratios=(0,) instead of penalty='l2'  and l1_ratios=(1,) instead of penalty='l1'.
  warnings.warn(
/usr/local/python/3.12.1/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1823: FutureWarning: The fitted attributes of LogisticRegressionCV will be simplified in scikit-learn 1.10 to remove redundancy. Set`use_legacy_attributes=

Best C selected by cross-validation: 10000.000000
Note: high C — model needed little regularisation. TF-IDF restriction is doing the work.


---
## Step 9 — Evaluate on held-out test set

In [12]:
y_pred = model.predict(X_test)

print('=== Classification Report (held-out test set) ===')
print(classification_report(y_test, y_pred, target_names=['Human', 'AI']))
print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}')
print()
print('Confusion Matrix:')
print(confusion_matrix(y_test, y_pred))

=== Classification Report (held-out test set) ===
              precision    recall  f1-score   support

       Human       0.99      0.98      0.99      4489
          AI       0.99      0.99      0.99      4661

    accuracy                           0.99      9150
   macro avg       0.99      0.99      0.99      9150
weighted avg       0.99      0.99      0.99      9150

Accuracy: 0.9860

Confusion Matrix:
[[4419   70]
 [  58 4603]]


---
## Step 10 — 5-fold cross-validation on full dataset

### Why this matters

A single 80/20 split gives one number. Cross-validation gives you:
- **Mean score** — the expected real-world performance
- **Std score** — how much performance varies across different data subsets

If mean ≈ 0.90 and std < 0.02, the model is stable and likely generalises well.
If std > 0.05, the model is sensitive to which data it sees — a warning sign.

**Important:** we re-assemble the full sparse matrix to do CV over the entire dataset.
In a strict pipeline this would require wrapping everything in `sklearn.Pipeline` to
prevent leakage — here we report CV as a diagnostic, not as the final evaluation.
The test set result above (Step 9) is the proper held-out evaluation.

In [13]:
# Re-assemble full feature matrix for CV reporting
X_all_tfidf = tfidf.transform(X_ans)                               # already fitted on train
X_all_custom = X_custom.replace([np.inf, -np.inf], 0).fillna(0)
X_all_custom_scaled = scaler.transform(X_all_custom)               # already fitted on train
X_all = hstack([X_all_tfidf, X_all_custom_scaled])

# Use a fixed-C model for CV scoring (using best C from LogisticRegressionCV)
from sklearn.linear_model import LogisticRegression
cv_model = LogisticRegression(
    C=best_C,
    penalty='l2',
    solver='saga',
    max_iter=2000,
    class_weight='balanced',
    random_state=RANDOM_STATE,
)

cv_scores = cross_val_score(
    cv_model, X_all, y,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    scoring='f1_macro',
    n_jobs=-1
)

print('=== 5-Fold Stratified Cross-Validation (F1-macro) ===')
for i, s in enumerate(cv_scores, 1):
    print(f'  Fold {i}: {s:.4f}')
print(f'\nMean: {cv_scores.mean():.4f}  ±  {cv_scores.std():.4f}')

gap = abs(accuracy_score(y_test, y_pred) - cv_scores.mean())
print(f'\nGap (test accuracy vs CV mean F1): {gap:.4f}')
if gap > 0.05:
    print('WARNING: Gap > 5%. The model may still be overfitting to the test split.')
else:
    print('OK: Gap is small — performance is consistent across splits.')

/usr/local/python/3.12.1/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/usr/local/python/3.12.1/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/usr/local/python/3.12.1/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.w

=== 5-Fold Stratified Cross-Validation (F1-macro) ===
  Fold 1: 0.9841
  Fold 2: 0.9860
  Fold 3: 0.9832
  Fold 4: 0.9854
  Fold 5: 0.9820

Mean: 0.9841  ±  0.0015

Gap (test accuracy vs CV mean F1): 0.0019
OK: Gap is small — performance is consistent across splits.


/usr/local/python/3.12.1/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


---
## Step 11 — Save artifacts

In [14]:
feature_cols_saved = X_custom_train.columns.tolist()

joblib.dump(model,             'logistic_model.pkl')
joblib.dump(tfidf,             'tfidf_vectorizer.pkl')
joblib.dump(scaler,            'feature_scaler.pkl')
joblib.dump(feature_cols_saved,'feature_columns.pkl')

print('Saved:')
print('  logistic_model.pkl')
print('  tfidf_vectorizer.pkl')
print('  feature_scaler.pkl')
print('  feature_columns.pkl')
print(f'  Feature columns order: {feature_cols_saved}')

Saved:
  logistic_model.pkl
  tfidf_vectorizer.pkl
  feature_scaler.pkl
  feature_columns.pkl
  Feature columns order: ['chat_word_ratio', 'punct_ratio', 'ttr', 'function_word_ratio', 'discourse_ratio', 'sentence_length_mean', 'sentence_length_std', 'avg_word_length']


---
## Step 12 — Inference on new text

### Key fix: no chunking — mirror training exactly

The original inference split text into 3-sentence chunks, extracted features per chunk,
and averaged probabilities. But **training was done on whole answers**, so the feature
distributions at inference (computed on 3 sentences) were completely different from
those at training (computed on 15+ sentences). For example, `sentence_length_std` on
a 3-sentence chunk is near-zero; on a full answer it is meaningful.

**Fixed**: pass the full text, extract features once, predict once.
This matches exactly what the model saw during training.

In [15]:
def predict_text(text, model, tfidf, scaler, feature_cols):
    """
    Predict whether a piece of text is AI-generated (1) or human-written (0).
    Mirrors training exactly: full-text feature extraction, single prediction.
    """
    # 1. Clean text
    text_clean = clean_text(text.lower())

    # 2. Extract custom features from full text (same as training)
    sentences = split_sentences(text_clean)
    feats     = aggregate_sentence_features(sentences)

    feat_df = pd.DataFrame([feats])
    feat_df = feat_df.replace([np.inf, -np.inf], 0).fillna(0)
    feat_df = feat_df.reindex(columns=feature_cols, fill_value=0)

    # 3. Scale custom features
    custom_scaled = scaler.transform(feat_df)

    # 4. TF-IDF transform
    text_tfidf = tfidf.transform([text_clean])

    # 5. Fuse and predict
    X = hstack([text_tfidf, custom_scaled])
    prob_ai    = model.predict_proba(X)[0, 1]
    prediction = 1 if prob_ai > 0.5 else 0

    return {
        'prediction'   : 'AI' if prediction == 1 else 'Human',
        'prob_ai'      : round(prob_ai, 4),
        'prob_human'   : round(1 - prob_ai, 4),
    }


# --- Demo ---
sample_ai = """In this pipeline, I am performing data cleaning before passing features 
into the model. First, I replace all missing values (NaN) with 0 to ensure there are 
no undefined values in the dataset. Then, I handle infinite values by replacing them 
with 0, since such values can occur due to division operations in feature engineering. 
Finally, I align the feature DataFrame with the exact column order used during training 
by reindexing using the saved feature list. This ensures that the model receives input 
in the same structure it was trained on, which is critical for correct predictions."""

sample_human = """honestly i just googled it lol. the way i think about it is like, 
imagine you're trying to split a pizza — if you can't do it evenly then you need to 
figure out what the common factor is. my teacher explained it way better but basically 
that's the gist of it. might be wrong tho haha"""

print('=== Inference Demo ===')
print()
result_ai = predict_text(sample_ai, model, tfidf, scaler, feature_cols_saved)
print(f'Sample (expected AI):    {result_ai}')

result_human = predict_text(sample_human, model, tfidf, scaler, feature_cols_saved)
print(f'Sample (expected Human): {result_human}')

=== Inference Demo ===

Sample (expected AI):    {'prediction': 'Human', 'prob_ai': np.float64(0.0412), 'prob_human': np.float64(0.9588)}
Sample (expected Human): {'prediction': 'Human', 'prob_ai': np.float64(0.0141), 'prob_human': np.float64(0.9859)}
